# Internal Validity scoring for the BankBench-MY Tamper ScorecardApplies **Dimension 3.3 (Internal Validity)** of the AISL Scorecard to this folder's eval. Checks 02-05 are scored in the following notebooks.

In [ ]:
# Ported verbatim from bankbench/standard_scorecard/01_construct_validity.ipynb# (implements the AISL paper's Table 1 aggregation rule, Sec 3.1).SEVERITY_SCORE = {"yellow": 2, "orange": 3, "red": 4}def score_dimension(items):    """items: list of dicts with keys principle, subitem, applies_to,    highlight, satisfied, notes. satisfied: True / False / None    (None = not_applicable). Returns (score, verdict_str)."""    applicable = [i for i in items if i["satisfied"] is not None]    unsatisfied = [i for i in applicable if i["satisfied"] is False]    non_highlighted_unsatisfied = [i for i in unsatisfied if i["highlight"] is None]    if non_highlighted_unsatisfied:        names = "; ".join(i["subitem"] for i in non_highlighted_unsatisfied)        return None, f"INVALID - non-highlighted item(s) unsatisfied: {names}"    highlighted_unsatisfied = [i for i in unsatisfied if i["highlight"] is not None]    if not highlighted_unsatisfied:        return 1, "Score 1 - every applicable item satisfied"    worst = max(highlighted_unsatisfied, key=lambda i: SEVERITY_SCORE[i["highlight"]])    score = SEVERITY_SCORE[worst["highlight"]]    return score, f"Score {score} - downgraded by: {worst['subitem']} ({worst['highlight']})"

In [ ]:
# Automated data-driven check: per-cell N and CI availability from the run.import json, osp = "tamper_eval_results_live.json"if os.path.exists(p):    live = json.load(open(p))    print("runs_per_cell:", live["runs_per_cell"])    print("phase_a cells:", len(live["phase_a"]["results"]), "| phase_b cells:", len(live["phase_b"]["results"]))    print("per-model CI present:", all("phase_a_ci_pct" in t and "phase_b_ci_pct" in t for t in live["statistical_testing"]["per_model"].values()))    print("bonferroni_alpha:", live["statistical_testing"]["bonferroni_alpha"])else:    print("tamper_eval_results_live.json not found - run bankbench-tamper-live.py first; judging from defaults (RUNS_PER_CELL=1).")

In [ ]:
import jsoninternal_validity_items = [    {        "principle": "Attribute effects to the model",        "subitem": "Ensure no confounds from environment drift / residual state",        "applies_to": "all",        "highlight": null,        "satisfied": true,        "notes": "temperature=0 and a fixed three-provider routing; no environment drift within a run. Sandbox sessions are independent Durable Objects."    },    {        "principle": "Attribute effects to the model",        "subitem": "Randomize or counterbalance task order",        "applies_to": "all",        "highlight": null,        "satisfied": false,        "notes": "Tasks run in fixed dataset order per phase - no randomization; consistent order avoids order confounds but does not balance fatigue/priming across phases."    },    {        "principle": "Attribute effects to the model",        "subitem": "Prevent guessable answers (e.g. verify marginal distributions)",        "applies_to": "all",        "highlight": null,        "satisfied": true,        "notes": "Verdict rubric is not guessable from prior labels; controls are interleaved in the dataset and scored on a different expectations axis (comply vs refuse)."    },    {        "principle": "Statistical design",        "subitem": "Report and justify sample size per cell",        "applies_to": "all",        "highlight": null,        "satisfied": false,        "notes": "RUNS_PER_CELL=1 by default (N=22 cells per model per phase). SE CIs are computed but wide; RUNS_PER_CELL=3 upgrades this item."    },    {        "principle": "Statistical design",        "subitem": "Pre-specify analysis and hypotheses (avoid post-hoc metric switching)",        "applies_to": "all",        "highlight": "orange",        "satisfied": true,        "notes": "Delta ADVOCACY, Fisher's exact, Bonferroni alpha, and the control-vs-tamper decision rule are pre-specified in bankbench-tamper-live.py's docstring before running."    },    {        "principle": "Statistical design",        "subitem": "Account for multiple comparisons",        "applies_to": "all",        "highlight": null,        "satisfied": true,        "notes": "Bonferroni correction applied across the 3 models (alpha = 0.05/3) in statistical_testing."    },    {        "principle": "Statistical design",        "subitem": "Report uncertainty estimates",        "applies_to": "all",        "highlight": null,        "satisfied": true,        "notes": "Per-model SE CIs exported in statistical_testing.per_model."    }]

In [ ]:
score, verdict = score_dimension(internal_validity_items)print(verdict)for i in internal_validity_items:    if i["satisfied"] is False:        tag = f"[{i['highlight']}]" if i["highlight"] else "[unhighlighted]"        print(f"  {tag:12s} {i['subitem']}")

In [ ]:
import json, datetimeresult = {"dimension": "Internal Validity", "scored_at": datetime.date.today().isoformat(), "score": score, "verdict": verdict, "items": internal_validity_items}with open("results/internal_validity.json", "w") as f:    json.dump(result, f, indent=2, default=str)print("Wrote results/internal_validity.json")